In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Imports

In [2]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

import matplotlib.pyplot as plt

import copy
import random
import time

import sys

from kaggle_environments import make

from IPython.display import clear_output, display

00:37:23 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [Errno -3] Temporary failure in name resolution. Falling back to local backup.


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

## Architecture

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ACTION_NAMES = [
    "NORTH", "SOUTH", "EAST", "WEST", "IDLE", # general movement
    "BUILD_SCOUT", "BUILD_WORKER", "BUILD_MINER", # factory
    "JUMP_NORTH", "JUMP_SOUTH", "JUMP_EAST", "JUMP_WEST", # factory
    "BUILD_NORTH", "BUILD_SOUTH", "BUILD_EAST", "BUILD_WEST", # worker
    "REMOVE_NORTH", "REMOVE_SOUTH", "REMOVE_EAST", "REMOVE_WEST", # worker
    "TRANSFORM", # miner
    "TRANSFER_NORTH", "TRANSFER_SOUTH", "TRANSFER_EAST", "TRANSFER_WEST" # all robots
]

class CentralizedMARL(nn.Module):
    def __init__(self, map_channels=19, robot_features=11, embed_dim=128, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        
        # 1. The Actor's CNN (Learning to See)
        self.cnn = nn.Sequential(
            nn.Conv2d(map_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, embed_dim, kernel_size=3, padding=1), # Expands to 128
            nn.ReLU()
        )
        
        # 2. Robot Feature Projector
        self.robot_proj = nn.Linear(robot_features, embed_dim)
        
        # 3. Transformer (Learning to Route Attention)
        # Splits 128 dims into 4 heads of 32 dims [cite: 524]
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        
        # 4. The Actor's Policy Head (Choosing Moves)
        self.policy_head = nn.Linear(embed_dim, len(ACTION_NAMES))
        
        # 5. The Centralized Critic Head (Appraising the Board)
        # The Critic looks at the flattened global map and outputs a single Value score
        self.critic_head = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, map_tensor, robot_tensor, action_masks=None, actions=None):
        map_tensor = map_tensor.to(device)
        robot_tensor = robot_tensor.to(device)

        if action_masks is not None:
            action_masks = action_masks.to(device)
        if actions is not None:
            actions = actions.to(device)
        
        B = map_tensor.size(0)
        
        # CNN Spatial Extraction
        map_features = self.cnn(map_tensor) # [B, 128, 20, 20]
        
        # Tokenization for Transformer
        map_tokens = map_features.permute(0, 2, 3, 1).flatten(1, 2) # [B, 400, 128]
        
        # Critic Value Calculation (Uses global map)
        global_map_summary = map_tokens.mean(dim=1)
        state_value = self.critic_head(global_map_summary) # [B, 1]
        
        # Actor Attention Routing
        if robot_tensor.size(1) > 0: # If robots are alive
            queries = self.robot_proj(robot_tensor) # [B, N, 128]
            attn_out, _ = self.attention(query=queries, key=map_tokens, value=map_tokens)
            
            # Action Selection
            logits = self.policy_head(attn_out) # [B, N, 25]
            
            if action_masks is not None:
                logits = logits.masked_fill(~action_masks, -1e9)
                
            dist = Categorical(logits=logits)

            if actions is None:
                actions = dist.sample()
                
            log_probs = dist.log_prob(actions)
            entropy = dist.entropy()
        else:
            N_dim = robot_tensor.size(1)
            actions = torch.empty(B, N_dim, dtype=torch.long, device=device)
            log_probs = torch.empty_like(actions, dtype=torch.float)
            entropy = torch.zeros_like(actions, dtype=torch.float)
            
        return actions, log_probs, state_value, entropy

## Forward Pass

In [4]:
def generate_legal_mask(obs, active_uids):
    # ACTION_NAMES indices:
    # 0-3: Move, 4: Idle, 5-7: Build Units, 8-11: Jump
    # 12-15: Build Wall, 16-19: Remove Wall, 20: Transform, 21-24: Transfer
    
    N = len(active_uids)
    mask = torch.zeros((1, N, 25), dtype=torch.bool)
    
    for i, uid in enumerate(active_uids):
        r_type = obs['robots'][uid][0] # 0:Factory, 1:Scout, 2:Worker, 3:Miner
        
        # Everyone can Idle (4)
        mask[0, i, 4] = True
        
        if r_type == 0: # Factory
            mask[0, i, 5:12] = True  # Can Build Units and Jump
            mask[0, i, 21:25] = True # Can Transfer Energy
        elif r_type == 1: # Scout
            mask[0, i, 0:4] = True   # Move only
            mask[0, i, 21:25] = True # Can Transfer Energy
        elif r_type == 2: # Worker
            mask[0, i, 0:4] = True   # Move
            mask[0, i, 12:20] = True # Build/Remove Walls
            mask[0, i, 21:25] = True # Transfer Energy
        elif r_type == 3: # Miner
            mask[0, i, 0:4] = True   # Move
            mask[0, i, 20] = True    # Transform
            mask[0, i, 21:25] = True # Transfer Energy
            
    return mask

def process_state(obs, config, player_id=0):
    """
    Converts Kaggle dictionary into a 6-channel tensor and a robot feature matrix.
    """
    # 1. Parse Dimensions
    W, H = config['width'], config['height']
    sb = obs['southBound']
    walls = np.array(obs.walls, dtype=np.int32).reshape(H, W)
    
    n_friendly_robots = 0
    n_enemy_robots = 0
    
    active_uids = []
    
    # Build 18 Channels
    channels = np.zeros((19, H, W), dtype=np.float32)

    '''
    Channels:
        0.  isUndiscovered
        1.  hasNWall
        2.  hasEWall
        3.  hasSWall
        4.  hasWWall
        5.  isMyFactory
        6.  isMyScout
        7.  isMyWorker
        8.  isMyMiner
        9.  isEnemyFactory
        10. isEnemyScout
        11. isEnemyWorker
        12. isEnemyMiner
        13. myRobotsEnergy
        14. enemyRobotsEnergy
        15. crystalEnergy
        16. isMineNode
        17. mineOwner
        18. southboundHeatMap
    '''

    # MY LOGIC

    # Channel 0 : isUndiscovered
    channels[0] = np.where(walls == -1, 1, 0)

    # Channels 1-4 : has walls NSEW
    walls = np.where(walls == -1, 0, walls)
    channels[1] = (walls >> 0) & 1 # N
    channels[2] = (walls >> 1) & 1 # E
    channels[3] = (walls >> 2) & 1 # S
    channels[4] = (walls >> 3) & 1 # W

    # Channels 5-14 [locations of my robots, enemy robots, and energies of all robots]
    for uid, data in obs.robots.items() :
        robot_type, col, row, energy, owner = data[0], data[1], data[2], data[3], data[4]
        row = row - sb

        # tracking for later
        if owner==player_id: 
            active_uids.append(uid)
            n_friendly_robots = n_friendly_robots + 1
            # print("Counting friendly robots")
        else: n_enemy_robots = n_enemy_robots + 1

        if row<0 or row>=H or col<0 or col>=W:
            continue

        if owner == player_id: # my robot
            # Channel 13 : myRobotsEnergy
            channels[13][row][col] = energy / 1000
            if robot_type == 0: # Channel 5 : isMyFactory
                channels[5][row][col] = 1
            elif robot_type == 1: # Channel 6 : isMyScout
                channels[6][row][col] = 1
            elif robot_type == 2: # Channel 7 : isMyWorker
                channels[7][row][col] = 1
            elif robot_type == 3: # Channel 8 : isMyMiner
                channels[8][row][col] = 1
        else: # enemy robot
            # Channel 14 : enemyRobotsEnergy
            channels[14][row][col] = energy / 1000
            if robot_type == 0: # Channel 9 : isEnemyFactory
                channels[9][row][col] = 1
            elif robot_type == 1: # Channel 10 : isEnemyScout
                channels[10][row][col] = 1
            elif robot_type == 2: # Channel 11 : isEnemyWorker
                channels[11][row][col] = 1
            elif robot_type == 3: # Channel 12 : isEnemyMiner
                channels[12][row][col] = 1

    # Channel 15 : crystalEnergy
    for pos_str, energy in obs.crystals.items():
        col, row = map(int, pos_str.split(','))
        row = row - sb
        if row<0 or row>=H or col<0 or col>=W: continue
        channels[15][row][col] = energy / 50

    # Channel 16 : isMineNode
    for pos_str, isNode in obs.miningNodes.items():
        col, row = map(int, pos_str.split(','))
        row = row - sb
        if row<0 or row>=H or col<0 or col>=W: continue
        channels[16][row][col] = 1

    # Channel 17 : mineOwner
    for pos_str, data in obs.mines.items():
        energy, _, owner = data[0], data[1], data[2]
        col, row = map(int, pos_str.split(','))
        row = row - sb
        if row<0 or row>=H or col<0 or col>=W: continue
        if owner == player_id:
            channels[17][row][col] = 1
        elif owner == -1:
            channels[17][row][col] = -1

    # Channel 18 : southboundHeatMap
    for row in range(H):
        channels[18][row, :] = row / H


    '''
    Robot features :
        0.  row
        1.  col
        2.  normalised energy
        3.  isFactory
        4.  isScout
        5.  isWorker
        6.  isMiner
        7.  move_cd
        8.  jump_cd
        9.  build_cd
        10. owner
    '''
    robot_features = np.zeros((n_friendly_robots, 11), dtype=np.float32)
    idx = 0

    for uid, data in obs.robots.items() :
        robot_type, col, row, energy, owner, move_cd, jump_cd, build_cd = data[0], data[1], data[2], data[3], data[4], data[5], data[6], data[7]
        row = row - sb

        if owner!=player_id or row<0 or row>=H or col<0 or col>=W:
            continue

        robot_features[idx][0] = row
        robot_features[idx][1] = col
        robot_features[idx][2] = energy / 1000
        robot_features[idx][3] = 1 if robot_type == 0 else 0
        robot_features[idx][4] = 1 if robot_type == 1 else 0
        robot_features[idx][5] = 1 if robot_type == 2 else 0
        robot_features[idx][6] = 1 if robot_type == 3 else 0
        robot_features[idx][7] = move_cd
        robot_features[idx][8] = jump_cd
        robot_features[idx][9] = build_cd
        robot_features[idx][10] = 1 if owner==player_id else -1

        idx = idx + 1

    
    map_tensor = torch.tensor(channels).unsqueeze(0).to(device) # [1, 6, 20, 20]
    robot_tensor = torch.tensor(robot_features).unsqueeze(0).to(device) # [1, N, 4]
    
    # (Placeholder) Generate legal mask of size [1, N, 25] based on unit types
    # action_masks = torch.ones((1, len(active_uids), 25), dtype=torch.bool)
    action_masks = generate_legal_mask(obs, active_uids)
    
    return map_tensor, robot_tensor, action_masks, active_uids

## Backward Pass

In [5]:
class PPOAgent:
    def __init__(self, model, lr=3e-4, gamma=0.99, clip_ratio=0.2):
        self.model = model
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.gamma = gamma
        self.clip_ratio = clip_ratio

    def update(self, memory):
        # 1. Unpack Rollout Memory
        states_map = torch.stack([m.squeeze(0) for m in memory['maps']]).to(device)

        robot_list = [r.squeeze(0) for r in memory['robs']]
        states_rob = pad_sequence(robot_list, batch_first=True, padding_value=0.0).to(device)

        masks_list = [m.squeeze(0) for m in memory['masks']]
        states_mask = pad_sequence(masks_list, batch_first=True, padding_value=False).to(device)
        
        actions_list = [a.squeeze(0) for a in memory['acts']]
        old_actions = pad_sequence(actions_list, batch_first=True, padding_value=0).to(device)

        log_probs_list = [lp.squeeze(0) for lp in memory['log_probs']]
        old_log_probs = pad_sequence(log_probs_list, batch_first=True, padding_value=0.0).to(device)
        
        rewards = memory['rewards'] if 'rewards' in memory else memory['rews']
        
        # 2. Calculate Discounted Returns (Accumulated Reward)
        returns = []
        discounted_sum = 0
        for r in reversed(rewards):
            discounted_sum = float(r) + (self.gamma * discounted_sum)
            returns.insert(0, discounted_sum)
        returns = torch.tensor(returns).float().to(device)
        
        # 3. Forward Pass for current Values and Probabilities
        _, new_log_probs, state_values, entropy = self.model(states_map, states_rob, action_masks=states_mask, actions=old_actions)
        
        # 4. Calculate Advantage
        advantages = returns.unsqueeze(1) - state_values.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        advantages_expanded = advantages.expand(-1, new_log_probs.size(1))

        robot_padding_mask = states_mask.any(dim=-1)
        
        # 5. PPO Policy Loss (The Clipped "Blame Game")
        ratio = torch.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantages_expanded
        surr2 = torch.clamp(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio) * advantages_expanded
        actor_loss = -torch.min(surr1, surr2).mean()

        actor_loss = -torch.min(surr1, surr2)[robot_padding_mask].mean()
        # 6. Critic Loss (How wrong was the appraiser?)
        critic_loss = F.mse_loss(state_values, returns.unsqueeze(1))
        
        # 7. Total Loss + Entropy (Encourages exploration)
        loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy[robot_padding_mask].mean()
        
        # 8. Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

## Training loop

In [6]:
def calculate_dense_reward(obs, prev_energy):
    """
    Calculates R_t = Delta Energy + Alpha(Surviving) - Beta(Factory Death)
    """
    current_energy = sum([r[3] for r in obs.robots.values() if int(r[4]) == int(obs.player)])
    delta_energy = current_energy - prev_energy
    
    # Check if factory is alive
    factory_alive = any([r[0] == 0 for r in obs.robots.values() if int(r[4]) == int(obs.player)])
    
    reward = (delta_energy * 0.01) + 0.1 # Alpha (Survival step)
    if not factory_alive:
        reward -= 100.0 # Beta (Massive Death Penalty)

    return reward, current_energy

win_rates_against_random = []
loss_rates_against_random = []
draw_rates_against_random = []
win_rates_against_previous_opponent = []
loss_rates_against_previous_opponent = []
draw_rates_against_previous_opponent = []
state_dicts = []

def train_loop_random(env, active_agent, episodes=1000, gameplay_record=False):
    for ep in range(episodes):
        # 1. Setup the Match
        state_p0, state_p1 = env.reset()

        if np.random.random() > 0.5: player_idx, opp_idx = 0, 1
        else: opp_idx, player_idx = 0, 1
        
        memory = {'maps': [], 'robs': [], 'acts': [], 'log_probs': [], 'rews': [], 'masks': []}
        prev_energy = 1000
        done = False
        
        while not done:
            # Check for terminal states before processing
            if state_p0['status'] in ["DONE", "ERROR", "TIMEOUT", "INVALID"]:
                # print(f"Player 0 status: {state_p0['status']}. Episode over")
                done = True
                break
                
            # Check for terminal states before processing
            if state_p1['status'] in ["DONE", "ERROR", "TIMEOUT", "INVALID"]:
                # print(f"Player 1 status: {state_p0['status']}")
                done = True
                break

            state_p1['observation']['southBound'] = state_p0['observation']['southBound']

            player_state = state_p0 if player_idx==0 else state_p1
                
            # 2. Extract observations
            player_obs = player_state['observation']
            
            # --- ACTIVE PLAYER (Player 0) ---
            # Process state and generate actions
            map_t, rob_t, masks, active_uids = process_state(player_obs, env.configuration, player_id=player_idx)
            
            # Ensure model is on the correct device
            actions, log_probs, _, _ = active_agent.model(map_t, rob_t, masks)
            
            # Flatten actions to avoid indexing errors
            flat_actions = actions.view(-1)
            player_action_dict = {uid: ACTION_NAMES[flat_actions[i].item()] for i, uid in enumerate(active_uids)}
            
            # --- OPPONENT (Player 1) ---
            # Opponent plays random actions automatically by passing None or raw random logic
            # Depending on your specific env setup, passing None or a dedicated random dict:
            joint_actions = [player_action_dict, None] if player_idx==0 else [None, player_action_dict]
            
            # 3. Step the environment
            state_p0, state_p1 = env.step(joint_actions)
            
            # 4. Calculate dense reward
            dense_reward, prev_energy = calculate_dense_reward(player_state['observation'], prev_energy)
            dense_reward = float(dense_reward)
            
            # 5. Store memory (Move to CPU for storage)
            memory['maps'].append(map_t.cpu())
            memory['robs'].append(rob_t.cpu())
            memory['acts'].append(actions.cpu())
            memory['log_probs'].append(log_probs.cpu())
            memory['rews'].append(dense_reward)
            memory['masks'].append(masks.cpu())
            
        # 6. Update the active brain
        if len(memory['rews']) > 0:
            active_agent.update(memory)

        # 7. Checkpointing
        if ep % 50 == 0:
            torch.save(active_agent.model.state_dict(), "maze_brain.pth")
            # print(f"Episode {ep:>5}: Model saved")

        if gameplay_record == True and ep % 100 == 0:
            start_time = time.perf_counter()
            state_dicts.append(active_agent.model.state_dict())

            # Against random
            win_rate_against_random, loss_rate_against_random, draw_rate_against_random = play_games(n_games=100, opp_agent="random")
            win_rates_against_random.append(win_rate_against_random)
            loss_rates_against_random.append(loss_rate_against_random)
            draw_rates_against_random.append(draw_rate_against_random)
            print(f"Random-move player | Win rate = {win_rate_against_random:.2f} | Draw rate = {draw_rate_against_random:.2f} | Loss rate = {loss_rate_against_random:.2f} | Time = {time.perf_counter()-start_time}")
        

def train_loop_self_play(env, active_agent, episodes=1000, gameplay_record=False):
    # The opponent pool starts with a copy of the untrained random brain
    opponent_pool = [copy.deepcopy(active_agent.model)]
    
    for ep in range(episodes):
        # 1. Setup the Match
        state_p0, state_p1 = env.reset()

        if np.random.random() > 0.5: player_idx, opp_idx = 0, 1
        else: opp_idx, player_idx = 0, 1
        
        # 2. Select an opponent from history (usually the most recent one)
        opponent_model = random.choice(opponent_pool[-5:]) # Pick from last 5 versions
        opponent_model.eval() # Opponent does not learn
        
        memory = {'maps': [], 'robs': [], 'acts': [], 'log_probs': [], 'rews': [], 'masks': []}
        prev_energy = 1000
        done = False
        
        while not done:
            # Check for terminal states before processing
            if state_p0['status'] in ["DONE", "ERROR", "TIMEOUT", "INVALID"]:
                # print(f"Player 0 status: {state_p0['status']}. Episode over")
                done = True
                break
                
            # Check for terminal states before processing
            if state_p1['status'] in ["DONE", "ERROR", "TIMEOUT", "INVALID"]:
                # print(f"Player 1 status: {state_p0['status']}")
                done = True
                break

            state_p1['observation']['southBound'] = state_p0['observation']['southBound']

            player_state, opp_state = [state_p0, state_p1] if player_idx==0 else [state_p1, state_p0]
            player_obs = player_state['observation']
            opp_obs = opp_state['observation']
            
            # --- ACTIVE PLAYER (Player 0) ---
            map_t, rob_t, masks, active_uids = process_state(player_obs, env.configuration, player_id=player_idx)
            actions, log_probs, _, _ = active_agent.model(map_t, rob_t, masks)
            player_action_dict = {uid: ACTION_NAMES[actions[0][i].item()] for i, uid in enumerate(active_uids)}
            
            # --- OPPONENT (Player 1) ---
            # Process state from Player 1's perspective
            map_t_opp, rob_t_opp, masks_opp, opp_uids = process_state(opp_obs, env.configuration, player_id=opp_idx)
            with torch.no_grad(): # Opponent doesn't calculate gradients
                actions_opp, _, _, _ = opponent_model(map_t_opp, rob_t_opp, masks_opp)
            opp_action_dict = {uid: ACTION_NAMES[actions_opp[0][i].item()] for i, uid in enumerate(opp_uids)}
            
            # Combine actions and step the environment
            joint_actions = [player_action_dict, opp_action_dict] if player_idx==0 else [opp_action_dict, player_action_dict]
            state_p0, state_p1 = env.step(joint_actions)

            next_player_obs = player_state['observation']
            
            # Calculate dense reward ONLY for the active player
            dense_reward, prev_energy = calculate_dense_reward(next_player_obs, prev_energy)
            
            # Store memory for the active player
            memory['maps'].append(map_t)
            memory['robs'].append(rob_t)
            memory['acts'].append(actions)
            memory['log_probs'].append(log_probs)
            memory['rews'].append(dense_reward)
            memory['masks'].append(masks)
            
        # Update the active brain
        if len(memory['rews']) > 0:
            active_agent.update(memory)

        if ep % 10 == 0:
            torch.save(active_agent.model.state_dict(), "maze_brain.pth")
            # print(f"Episode {ep:>4}: Model saved")
        
        # Add the new brain to the opponent pool periodically
        if ep % 50 == 0:
            opponent_pool.append(copy.deepcopy(active_agent.model))
            # print(f"Episode {ep:>5}")

        if gameplay_record == True and ep % 100 == 0:
            state_dicts.append(active_agent.model.state_dict())

            # Against random
            start_time = time.perf_counter()
            win_rate_against_random, loss_rate_against_random, draw_rate_against_random = play_games(n_games=100, opp_agent="random")
            win_rates_against_random.append(win_rate_against_random)
            loss_rates_against_random.append(loss_rate_against_random)
            draw_rates_against_random.append(draw_rate_against_random)
            print(f"Random-move player | Win rate = {win_rate_against_random:.2f} | Draw rate = {draw_rate_against_random:.2f} | Loss rate = {loss_rate_against_random:.2f} | Time = {time.perf_counter()-start_time}")

            # Against previous opponent
            start_time = time.perf_counter()
            win_rate_against_previous_opponent, loss_rate_against_previous_opponent, draw_rate_against_previous_opponent = play_games(n_games=100, opp_agent=opponent_pool[-2 % len(opponent_pool)])
            win_rates_against_previous_opponent.append(win_rate_against_previous_opponent)
            loss_rates_against_previous_opponent.append(loss_rate_against_previous_opponent)
            draw_rates_against_previous_opponent.append(draw_rate_against_previous_opponent)
            print(f"Previous Opponent  | Win rate = {win_rate_against_previous_opponent:.2f} | Draw rate = {loss_rate_against_previous_opponent:.2f} | Loss rate = {draw_rate_against_previous_opponent:.2f} | Time = {time.perf_counter()-start_time}")

## Gameplay Testing

In [7]:
def play_games(n_games=100, opp_agent="random"):
    # print(f'Playing {n_games} against "{opp_agent}"')
    n_wins = 0
    n_draws = 0
    
    for game_idx in range(n_games):
        env = make("crawl", configuration={"episodeSteps": 500, "width": 20, "height": 20})
        
        if np.random.random()>=0.5:
            env.run([my_agent, opp_agent])
            player_idx = 0
            opp_idx = 1
        else:
            env.run([opp_agent, my_agent])
            player_idx = 1
            opp_idx = 0

        '''
        print()
        if player_idx == 0: print("MY AGENT                                                       OPPONENT")
        else: print("OPPONENT                                                       MY AGENT")
        env.render(mode="ipython", width=800, height=800)
        #'''
        
        my_agent_final = env.state[player_idx]
        opp_agent_final = env.state[opp_idx]
        my_status = my_agent_final['status']
        my_reward = my_agent_final['reward']
        opp_reward = opp_agent_final['reward']
    
        # Evaluate Win / Loss / Error Condition
        if my_status in ["ERROR", "TIMEOUT", "INVALID"]:
            print(f"Agent Lost due to code failure! Status: {my_status}")
        elif my_reward > opp_reward:
            # print(f"Agent WON! (Score: {my_reward} vs {opp_reward})")
            n_wins = n_wins + 1
        elif my_reward < opp_reward: pass
            # print(f"Agent LOST! (Score: {my_reward} vs {opp_reward})")
        else:
            # print(f"Game was a DRAW! (Score: {my_reward})")
            n_draws = n_draws + 1
    
        if game_idx % 5 == 0:
            # print(f"Game number {game_idx}")
            pass

    win_rate  = n_wins  / n_games
    loss_rate = (n_games - n_wins) / n_games
    draw_rate =  n_draws / n_games
    
    '''
    print(f"Win  rate = {win_rate}")
    print(f"Loss rate = {loss_rate}")
    print(f"Draw rate = {draw_rate}")
    '''

    return win_rate, loss_rate, draw_rate

## Deploy

In [8]:
# Global instantiation to prevent reloading model every step
trained_model = None

def my_agent(obs, config):
    global trained_model
    
    # 1. Load Model on first step
    if trained_model is None:
        trained_model = CentralizedMARL(map_channels=19, robot_features=11)
        # trained_model.load_state_dict(torch.load("marl_weights.pth"))

        weights_path = "/kaggle/working/maze_brain.pth" 
        
        if os.path.exists(weights_path):
            trained_model.load_state_dict(torch.load(weights_path))

        trained_model.eval() # Set to inference mode
        trained_model.to(device)
        
    # 2. Preprocess
    map_t, rob_t, masks, active_uids = process_state(obs, config, player_id=obs.player)
    
    # 3. Forward Pass (No gradients needed)
    with torch.no_grad():
        actions, _, _, _ = trained_model(map_t, rob_t, masks)

    # flat_actions = actions.view(-1)
        
    # 4. Format Output for Kaggle
    command_dict = {}
    if len(active_uids) > 0:
        for i, uid in enumerate(active_uids):
            chosen_action_idx = actions[0][i].item()
            command_dict[uid] = ACTION_NAMES[chosen_action_idx]
            
    return command_dict

In [9]:
n_episodes = 60_000

ep_counts = []
ep_runtimes = []
win_rates = []
loss_rates = []
draw_rates = []
state_dicts = []

start_time = time.perf_counter()

# 1. Initialize the Kaggle Environment
env = make("crawl", configuration={"episodeSteps": 500, "width": 20, "height": 20})
env.reset()

# 2. Initialize the Agent and Model
marl_model = CentralizedMARL()
marl_model.to(device)
agent = PPOAgent(marl_model)

# Load previous brain if it exists
if os.path.exists("maze_brain.pth"):
    # print("Loading existing brain")
    agent.model.load_state_dict(torch.load("maze_brain.pth"))
else: pass
    # print("No existing brain found. Starting from scratch")

# 3. Start the Training Loop
# print("Training...")
for _ in range(3):
    print("RANDOM")
    train_loop_random(env, agent, episodes=12_000, gameplay_record=True)

    print()
    print("SELF PLAY")
    train_loop_self_play(env, agent, episodes=8_000, gameplay_record=True)

run_time = time.perf_counter() - start_time
avg_ep_runtime = run_time / n_episodes
print(f"Time taken to train over {n_episodes} episodes = {run_time}")
print(f"Average episode runtime = {avg_ep_runtime : 3f}")

# Dynamic Plotting
'''
ep_counts.append((n_iter + 1) * n_episodes_per_train_game_iteration)
clear_output(wait=True)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Plot 1: Rates
ax1.plot(ep_counts, win_rates, label='Win Rate')
ax1.plot(ep_counts, loss_rates, label='Loss Rate')
ax1.plot(ep_counts, draw_rates, label='Draw Rate')
ax1.set_title('Agent Performance Over Episodes')
ax1.set_xlabel('Episodes')
ax1.set_ylabel('Rate')
ax1.legend()

# Plot 2: Runtime
ax2.plot(ep_counts, ep_runtimes, color='purple', label='Avg Runtime/Ep')
ax2.set_title('Training Speed')
ax2.set_xlabel('Episodes')
ax2.set_ylabel('Seconds')
ax2.legend()

plt.tight_layout()
display(fig)
plt.close(fig)
'''

RANDOM
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 27.237285790999977
Random-move player | Win rate = 0.01 | Draw rate = 0.00 | Loss rate = 0.99 | Time = 28.80978648700011
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 28.964659817999745
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 29.831922360000135
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 29.098929022999982
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 29.24527209100006
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 29.203355225999985
Random-move player | Win rate = 0.01 | Draw rate = 0.00 | Loss rate = 0.99 | Time = 27.540524690999973
Random-move player | Win rate = 0.00 | Draw rate = 0.00 | Loss rate = 1.00 | Time = 29.056185992999872
Random-move player | Win rate = 0.01 | Draw rate = 0.00 | Loss rate 

"\nep_counts.append((n_iter + 1) * n_episodes_per_train_game_iteration)\nclear_output(wait=True)\nfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))\n\n# Plot 1: Rates\nax1.plot(ep_counts, win_rates, label='Win Rate')\nax1.plot(ep_counts, loss_rates, label='Loss Rate')\nax1.plot(ep_counts, draw_rates, label='Draw Rate')\nax1.set_title('Agent Performance Over Episodes')\nax1.set_xlabel('Episodes')\nax1.set_ylabel('Rate')\nax1.legend()\n\n# Plot 2: Runtime\nax2.plot(ep_counts, ep_runtimes, color='purple', label='Avg Runtime/Ep')\nax2.set_title('Training Speed')\nax2.set_xlabel('Episodes')\nax2.set_ylabel('Seconds')\nax2.legend()\n\nplt.tight_layout()\ndisplay(fig)\nplt.close(fig)\n"